# Olvid Protocol Implementation

Olvid is a decentralized messaging protocol that does not rely on a central server for user discovery or key management. Instead, it uses a **Trust Establishment Protocol with SAS (Short Authentication String)** to securely exchange identities and establish a shared channel.

In [1]:
import hashlib
import hmac
import os
import time
import numpy as np
from cryptography.hazmat.primitives.asymmetric import x25519, ed25519
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

def sha256(data):
    return hashlib.sha256(data).digest()

## 1. Cryptographic Identity

In Olvid, an identity consists of public keys for encryption (Curve25519) and signing (Ed25519).

In [2]:
class OlvidIdentity:
    def __init__(self, name):
        self.name = name
        self.encryption_key = x25519.X25519PrivateKey.generate()
        self.signing_key = ed25519.Ed25519PrivateKey.generate()
        
    def get_public_identity(self):
        return {
            'name': self.name,
            'enc_pk': self.encryption_key.public_key(),
            'sig_pk': self.signing_key.public_key()
        }
    
    def bytes(self):
        # Simplified byte representation of identity
        return self.name.encode() + \
               self.encryption_key.public_key().public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw) + \
               self.signing_key.public_key().public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw)

alice_id = OlvidIdentity("Alice")
bob_id = OlvidIdentity("Bob")

## 2. Trust Establishment Protocol (SAS Handshake)

This protocol allows two users to exchange their identities and verify them using an 8-digit SAS code.

In [3]:
def compute_sas(seed_alice, seed_bob, identity_bob_bytes):
    # Simplified ComputeSAS procedure based on specifications
    h = hashlib.sha256(identity_bob_bytes + seed_alice).digest()
    # XOR seed_bob with hash (truncated to match length if necessary)
    combined = bytes(a ^ b for a, b in zip(h, seed_bob))
    # Generate 8-digit SAS
    sas_int = int.from_bytes(hashlib.sha256(combined).digest(), 'big') % 100000000
    return f"{sas_int:08d}"

print("Step 1: Alice generates seed and commitment")
seed_alice = os.urandom(32)
commitment_alice = sha256(seed_alice)

print("Step 2: Bob receives commitment, generates his seed and sends it")
seed_bob = os.urandom(32)

print("Step 3: Alice sends her seed (decommitment)")

print("Step 4: Both compute SAS and verify")
sas_alice = compute_sas(seed_alice, seed_bob, bob_id.bytes())
sas_bob = compute_sas(seed_alice, seed_bob, bob_id.bytes())

print(f"Alice's SAS: {sas_alice}")
print(f"Bob's SAS:   {sas_bob}")
assert sas_alice == sas_bob

Step 1: Alice generates seed and commitment
Step 2: Bob receives commitment, generates his seed and sends it
Step 3: Alice sends her seed (decommitment)
Step 4: Both compute SAS and verify
Alice's SAS: 42980775
Bob's SAS:   42980775


## 3. Metric Analysis

In [4]:
start_time = time.time()
id_temp = OlvidIdentity("Temp")
id_time = time.time() - start_time

start_time = time.time()
s1 = os.urandom(32)
s2 = os.urandom(32)
compute_sas(s1, s2, id_temp.bytes())
handshake_time = time.time() - start_time

print(f"Identity Generation Time: {id_time:.6f}s")
print(f"Handshake (SAS) Computation Time: {handshake_time:.6f}s")
print(f"Identity Size: {len(id_temp.bytes())} bytes")

Identity Generation Time: 0.000181s
Handshake (SAS) Computation Time: 0.000160s
Identity Size: 68 bytes


## 4. Security Analysis

### Decentralization and Metadata
Unlike Signal, Olvid does not have a central directory. This eliminates the "Discovery" metadata leak where a server knows who is registered. The SAS-based handshake ensures that even without a trusted server, Man-in-the-Middle (MitM) attacks are detectable by the users.

### Quantum Vulnerability
Olvid uses Curve25519 for its cryptographic identity and channel encryption. Similar to Signal, this is vulnerable to quantum attacks using Shor's algorithm. For a post-quantum version, these curves would need to be replaced with lattice-based KEMs and signatures.